In [1]:
# Load some jobs into a pandas dataframe

import pandas as  pd
pd.set_option('display.max_colwidth', 300)
pd.options.mode.chained_assignment = None


# replace with student spreadsheet

jobs = pd.read_csv('https://docs.google.com/spreadsheets/d/e/2PACX-1vRWmeWZs5NHptaUiotzWf8Qf9evBzdDToTvkMbn1v0ug5uqu32vPd4BwTU7DsogP9cU7xKov0ku8gYV/pub?output=csv')
jobs = jobs[['year', 'date', 'description']]
jobs.head(5)


,year,date,description
0,2026,04-02,"Director of Voter Services, Chester County, Pennsylvania– The Director of Voter Services plans, organizes, supervises, and manages the activities of voter registration, campaign finance, and elections for the County of Chester in compliance with all federal, state, county, and local laws. Essent..."
1,2026,8-Jan,"Account Manager, Electask– Electask is a fast-growing company transforming how election officials manage their operations. Our task and poll worker management software empowers election administrators – the backbone of democracy – to run elections more efficiently and with greater confidence. Ri..."
2,2026,8-Jan,"Election Software Specialist, Printelect– Printelect is a dynamic and well-established regional election technology & services company with a proven track record in ballot printing, mail services, and a diverse portfolio of election products. With over fifty employees and one hundred twenty year..."
3,2026,5-Feb,"Elections Division Manager, Washington County, Minnesota– Washington County is seeking a steady, collaborative, and resilient leader to serve as our Elections Division Manager. This role is well-suited for an experienced people manager who brings calm leadership to complex, high-pressure environ..."
4,2026,5-Feb,"Elections Specialist, Kalamazoo County, Michigan– The purpose of this job is to assist with the implementation and enforcement of Michigan election laws. Provide information and support to elected officials, members of the public, and local clerks regarding the election process. Implement Electi..."


In [ ]:
# let's try to extract information from the job description
jobs.loc[0, 'description']

'Elections Systems Administrator, Arapahoe County, Colorado – The Election Systems Administrator performs leadership level administrative, and professional work in carrying out a comprehensive public facing service operation. This position specifically leads and supports all areas of ballot processing, ballot auditing and election technology maintenance. Salary: $64,293.06 – $96,439.98 Annually. Deadline: April 16. Application: For the complete job listing and to apply, click here.\xa0'

In [ ]:
# let's get the salary.
# let's get the location
# let's get whether it's a nonprofit or a government position


def get_title(job_description):
    return job_description.split('–')[0]

def get_salary(job_description):
    return job_description.split('Salary: ')[1]

get_title(jobs.loc[0, 'description'])
# get_salary(jobs.loc[0, 'description'])

# this isn't gonna work!

'Elections Systems Administrator, Arapahoe County, Colorado '

In [ ]:
import requests
import os

# Replace 'YOUR_API_KEY' with your actual OpenWeatherMap API key
api_key = os.getenv('OPENWEATHER_API_KEY', 'YOUR_OPENWEATHER_API_KEY')

# Coordinates for Washington, DC
lat = 38.9072
lon = -77.0369

# One Call 3.0 API URL
url = f"https://api.openweathermap.org/data/3.0/onecall?lat={lat}&lon={lon}&appid={api_key}&units=imperial"

response = requests.get(url)
if response.status_code == 200:
    data = response.json()
    # Print current weather
    current = data['current']
    print(f"Temperature: {current['temp']}°F")
    print(f"Feels like: {current['feels_like']}°F")
    print(f"Humidity: {current['humidity']}%")
    print(f"Description: {current['weather'][0]['description']}")
else:
    print(f"Error: {response.status_code} - {response.text}")

44.91

In [13]:
import anthropic
from pydantic import BaseModel
from typing import List, Optional

client = anthropic.Anthropic(api_key="YOUR_ANTHROPIC_API_KEY")

class JobExtraction(BaseModel):
    job_title: str
    employer: str
    location: str
    salary_low_end: Optional[str] = None
    salary_high_end: Optional[str] = None
    skills: List[str]
    full_time: bool
    remote: bool
    nonprofit_or_government: str
    responsible_for_administering_elections: bool

def extract_job_info(job_description: str) -> JobExtraction:
    response = client.messages.parse(
        model="claude-opus-4-6",
        max_tokens=1024,
        messages=[
            {
                "role": "user",
                "content": f"Extract the key information from this job description: {job_description}"
            }
        ],
        output_format=JobExtraction,
    )
    return response.parsed_output

# Example usage
# job_info = extract_job_info(jobs.loc[0, 'description'])
# print(job_info)

In [19]:
from pydantic import BaseModel
from typing import List, Optional
import pandas as pd
from collections import Counter
from tqdm import tqdm

class SalarySkillsExtraction(BaseModel):
    salary_high_end: Optional[str] = None
    skills: List[str]

def extract_salary_skills(job_description: str):
    response = client.messages.parse(
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        messages=[
            {
                "role": "user",
                "content": f"Extract the highest salary mentioned and the key skills required from this job description: {job_description}"
            }
        ],
        output_format=SalarySkillsExtraction,
    )
    return response.parsed_output

# Process first 100 jobs
results = []
for i in tqdm(range(min(10, len(jobs))), desc="Processing jobs"):
    desc = jobs.loc[i, 'description']
    try:
        extracted = extract_salary_skills(desc)
        results.append({
            'index': i,
            'salary_high_end': extracted.salary_high_end,
            'skills': extracted.skills
        })
    except Exception as e:
        print(f"Error processing job {i}: {e}")
        results.append({
            'index': i,
            'salary_high_end': None,
            'skills': []
        })

# Convert to DataFrame for analysis
df_results = pd.DataFrame(results)

# Preliminary analysis
print("=== Salary Analysis ===")
salaries = df_results['salary_high_end'].dropna()
print(f"Number of jobs with salary info: {len(salaries)}")
if len(salaries) > 0:
    # Try to extract numeric values
    numeric_salaries = []
    for s in salaries:
        try:
            # Remove $ and commas, take first number
            num = ''.join(c for c in s if c.isdigit() or c == '.')
            if num:
                numeric_salaries.append(float(num))
        except:
            pass
    if numeric_salaries:
        print(f"Average high-end salary: ${sum(numeric_salaries)/len(numeric_salaries):.0f}")
        print(f"Min salary: ${min(numeric_salaries):.0f}")
        print(f"Max salary: ${max(numeric_salaries):.0f}")

print("\n=== Skills Analysis ===")
all_skills = [skill for skills_list in df_results['skills'] for skill in skills_list]
skill_counts = Counter(all_skills)
print(f"Total unique skills mentioned: {len(skill_counts)}")
print("Top 10 most common skills:")
for skill, count in skill_counts.most_common(10):
    print(f"  {skill}: {count}")

print(f"\nAverage skills per job: {len(all_skills)/len(df_results):.1f}")

Processing jobs: 100%|██████████| 10/10 [00:14<00:00,  1.42s/it]

=== Salary Analysis ===
Number of jobs with salary info: 9
Average high-end salary: $88673
Min salary: $46
Max salary: $200000

=== Skills Analysis ===
Total unique skills mentioned: 105
Top 10 most common skills:
  voter registration: 3
  team leadership: 2
  election planning: 2
  ballot processing: 2
  campaign finance: 1
  elections management: 1
  election operations: 1
  department leadership: 1
  administration: 1
  communications: 1

Average skills per job: 11.0
